#Build Agentic RAG Using LlamaIndex

### Load all the required libraries

In [1]:
!pip install -U -q llama-index llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface pypdf llama-index-readers-file

### Set OpenAI API Key

In [2]:
import os
os.environ["OPENAI_API_KEY"] = ""

### Login to HuggingFace

In [3]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN') # Access the secret
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token not found or notebook access not enabled.")

Successfully logged in to Hugging Face!


#Build Agentic RAG

###Load all the required libraries

In [4]:
!pip install --upgrade datasets
!pip install --upgrade huggingface-hub

###Load the input document

In [5]:
from llama_index.core import SimpleDirectoryReader

# Load document
reader = SimpleDirectoryReader(input_files=["/content/the-state-of-ai-how-organizations-are-rewiring-to-capture-value_final.pdf"])
documents = reader.load_data()
print(f"Loaded {len(documents)} document(s).")

Loaded 26 document(s).


###Split the data into chunks

In [6]:
from llama_index.core.node_parser import SentenceSplitter

# chunk_size of 1024 is a good default value
splitter = SentenceSplitter(chunk_size=1024)
# Create nodes from documents
nodes = splitter.get_nodes_from_documents(documents)

### Define the LLM and the Embedding Model

In [7]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# LLM model
Settings.llm = OpenAI(model="gpt-3.5-turbo")
# embedding model
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

### Create the vector index and summary index

In [8]:
from llama_index.core import SummaryIndex, VectorStoreIndex

# summary index
summary_index = SummaryIndex(nodes)
# vector store index
vector_index = VectorStoreIndex(nodes)

### Create the vector query engine and summary query engine

In [9]:
# summary query engine
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)

# vector query engine
vector_query_engine = vector_index.as_query_engine()

### Convert the vector and query engines into tools

In [10]:
from llama_index.core.tools import QueryEngineTool


summary_tool = QueryEngineTool.from_defaults(
    query_engine=summary_query_engine,
    description=(
        "Useful for summarization questions related to the State of AI document."
    ),
)

vector_tool = QueryEngineTool.from_defaults(
    query_engine=vector_query_engine,
    description=(
        "Useful for retrieving specific context from the State of AI document."
    ),
)

### Define a superset query engine

In [11]:
from llama_index.core.query_engine.router_query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector


query_engine = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[
        summary_tool,
        vector_tool,
    ],
    verbose=True
)

### Test whether the query engine works

In [12]:
response = query_engine.query("Who is Lareina Yee according to the document?")
print(str(response))

Selecting query engine 1: Useful for retrieving specific context from the State of AI document..
Lareina Yee is one of the authors of the document titled "The state of AI: How organizations are rewiring to capture value" along with Alex Singla, Alexander Sukharevsky, Michael Chui, and Bryce Hall.


### Convert the query engine into a tool

In [13]:
# Create tool wrapper around router
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="state_of_ai_report_assistant",
    description="Answers questions based on the McKinsey 2025 State of AI report.",
    return_direct=False,
)

### Define system prompt

In [14]:
system_prompt = """
You are a helpful assistant specialized in answering questions using the 'State of AI' March 2025 report by McKinsey.
Your task is to:

1. Use the Summary Tool when the user asks for high-level insights, trends, survey findings, or general understanding
   (e.g., "What are the top AI adoption practices?" or "Summarize the report's key findings").

2. Use the Vector Tool when the user is asking for specific statistics, organizational practices, exhibit-based
   evidence, or detailed examples
   (e.g., "What percentage of companies track AI KPIs?" or "What are the risks companies are mitigating?").

Refer only to the content of the report. If the user's query is outside this context, politely decline or redirect.

Check your answer multiple times to make sure it is actually relevant and mentioned in the document.

Examples of summary queries:
- "How are companies restructuring to adopt GenAI?"
- "What does the report say about workforce reskilling?"

Examples of specific/vector queries:
- "What percentage of companies have a roadmap for GenAI adoption?"
- "Who is responsible for AI governance in large firms?"

Always explain clearly, referencing exact statistics, frameworks, or concepts when relevant. Be concise and insightful.
"""

### Define the agent

In [15]:
from llama_index.core.agent.workflow import AgentWorkflow

query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=Settings.llm,
    system_prompt=system_prompt,
)

### Run the agent and analyze responses

In [16]:
question = "Who is Yareina Lee according to the document? Where is she mentioned in the document and in what context?"
response =  await query_engine_agent.run(question)
print(response)

Selecting query engine 1: The question 'Yareina Lee' is more likely to require retrieving specific context from the State of AI document rather than summarization..
Lareina Yee is the Global Managing Partner of McKinsey's QuantumBlack, a data analytics and AI subsidiary. She is mentioned in the report in the context of leading QuantumBlack, which focuses on AI and data analytics.


### Equip the agent with multiple tools

In [17]:
!pip install llama-index-tools-arxiv llama-index-tools-wikipedia duckduckgo-search
!pip install llama-index-tools-brave-search

### Add the new tools (ArXiV, Brave Search)

In [18]:
# Import additional tools
from llama_index.tools.arxiv import ArxivToolSpec
from llama_index.tools.wikipedia import WikipediaToolSpec
from llama_index.core.tools import QueryEngineTool
from llama_index.tools.brave_search import BraveSearchToolSpec

import requests
import json

In [19]:
# Create ArXiV tool

arxiv_tool = ArxivToolSpec()

arxiv_tools = arxiv_tool.to_tool_list()


# Create Brave Search tool

brave_search_tool_spec = BraveSearchToolSpec(api_key="")
brave_search_tools = brave_search_tool_spec.to_tool_list()


In [20]:
# Create enhanced agent with multiple tools
enhanced_tools = [query_engine_tool]  # Start with McKinsey report tool
enhanced_tools.extend(brave_search_tools)  # Add all brave search tools
enhanced_tools.extend(arxiv_tools)  # Add all arxiv tools

### Define the enhanced agent with all tools

In [21]:
# Create new enhanced agent
enhanced_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=enhanced_tools,
    llm=Settings.llm,
    system_prompt="""You are an AI research assistant with access to:
    1. The McKinsey 2025 State of AI report
    2. Web search capabilities
    3. ArXiv research paper search

    Use these tools to provide comprehensive, well-researched answers. When discussing AI trends,
    combine insights from the McKinsey report with recent research and web findings.""",
)

### Test agent with multiple questions - questions that can benefit from multiple tools

In [22]:
# Question 1: Combine McKinsey insights with recent research
question1 = """According to the McKinsey report, what are the main organizational changes companies are making for AI adoption?
Can you also search for recent research papers on AI governance and organizational transformation to provide additional context?"""

print("Question 1: Organizational changes and governance")
print("=" * 50)
response1 = await enhanced_agent.run(question1)
print(response1)
print("\n" + "="*80 + "\n")

Question 1: Organizational changes and governance
Selecting query engine 0: The question is asking for a summary of the main organizational changes companies are making for AI adoption, which aligns with the purpose of summarization questions related to the State of AI document..
Here are some recent research papers related to AI governance and organizational transformation:

1. **Distributed Attacks in Persistent-State AI Control**:
   - This paper discusses the dynamics of attacks in persistent-state AI control scenarios where coding agents distribute attacks across pull requests. It introduces the concept of Iterative VibeCoding for safely deploying AI and compares gradual and non-gradual attacks in this context.

2. **On the Emergence of Quantum Many-Body Chaos for Tunably-Broken Integrability**:
   - The paper presents a theory on the emergence of quantum many-body chaos as integrability is broken via a tunable parameter. It explores the transition from early-time integrable behav

In [23]:
# Question 2: Workflow Redesign and Implementation
question2 = """What does the McKinsey report say about workflow redesign for AI implementation?
Search ArXiv for papers on business process automation with AI and find current web articles about workflow transformation."""

print("Question 2: Workflow Redesign")
response2 =  await enhanced_agent.run(question2)
print(response2)

Question 2: Workflow Redesign
Selecting query engine 0: Workflow redesign for AI implementation may involve summarizing key points and recommendations from the State of AI document..
### ArXiv Papers on Business Process Automation with AI:
1. **[A framework to evaluate the viability of robotic process automation for business process activities](https://arxiv.org/pdf/2007.10900v2):**
   - This paper focuses on evaluating the viability of Robotic Process Automation (RPA) for automating business processes. It presents a framework with thirteen criteria to support practitioners in assessing RPA automation candidates.

2. **[Agentic Business Process Management Systems](https://arxiv.org/pdf/2601.18833v1):**
   - Discusses the evolution of Business Process Management (BPM) and the impact of Generative and Agentic Artificial Intelligence (AI) on process automation. It introduces an architectural vision for Agentic Business Process Management Systems (A-BPMS) that integrate autonomy, reasoning

In [24]:
# Question 3: Risk management and future trends
question3 = """Based on the McKinsey report, what are the key risks organizations are addressing with Gen AI?
Can you search the web for recent academic research on AI risk mitigation and compare with the report's findings?"""

print("Question 3: Risk management")
response3 = await enhanced_agent.run(question3)
print(response3)

Question 3: Risk management
Selecting query engine 0: The question is asking for a summary of the key risks organizations are addressing with Gen AI, which is more aligned with summarization questions related to the State of AI document..
### Recent Academic Research on AI Risk Mitigation:

1. **[A Generative AI Cybersecurity Risks Mitigation Model for Code Generation](https://www.nature.com/articles/s41598-025-34350-3)**
   - This research focuses on cybersecurity risks in automatic code generation using generative AI and presents solutions to mitigate these risks. It consolidates cybersecurity risks and generative AI mitigation practices found in recent literature within the past 5-10 years.

2. **[International AI Safety Report 2026](https://www.insideprivacy.com/artificial-intelligence/international-ai-safety-report-2026-examines-ai-capabilities-risks-and-safeguards/)**
   - The report emphasizes the importance of building societal resilience as a complement to technical safeguards

In [25]:
# Question that forces usage of all three tools
comprehensive_question = """Who is Lareina Yee in the McKinsey document and what are her views on AI's workforce impact?

After finding information about her from the document, please:
1. Search the web using Brave Search for recent articles, interviews, or news about Lareina Yee and her work on AI
2. Search ArXiv for any research papers she may have authored or co-authored related to AI, workforce transformation, or economic impact
3. Provide a comprehensive profile combining insights from all three sources about her expertise and contributions to AI research"""

print("Question: Comprehensive profile of Lareina Yee")
print("=" * 60)
print("This question should force the agent to use:")
print("1. Query Engine - to find info about Lareina Yee in the McKinsey document")
print("2. Brave Search - to find recent web articles/news about her")
print("3. ArXiv Search - to find any academic papers she's authored")
print("=" * 60)

response = await enhanced_agent.run(comprehensive_question)
print(response)

Question: Comprehensive profile of Lareina Yee
This question should force the agent to use:
1. Query Engine - to find info about Lareina Yee in the McKinsey document
2. Brave Search - to find recent web articles/news about her
3. ArXiv Search - to find any academic papers she's authored
Selecting query engine 1: The choice is more relevant for retrieving specific context from the State of AI document, which may be more suitable for finding information about Lareina Yee..
### Lareina Yee Profile:

#### McKinsey Global Institute Director:
Lareina Yee is a Senior Partner and the Director of the McKinsey Global Institute, known for her expertise in AI and its impact on the workforce. She has contributed significantly to understanding the implications of AI on the economy and workforce transformation.

#### Views on AI's Workforce Impact:
Lareina Yee emphasizes the importance of understanding and adapting to the changes brought by AI in the workforce. She highlights the need for employers t